<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Resposta:** Não é estritamente necessário normalizar os dados para modelos baseados em árvores (Random Forest e AdaBoost). Esses modelos são invariantes à escala das características, pois decidem as divisões baseando-se em limiares (thresholds) de cada feature individualmente. Contudo, em imagens, a normalização para o intervalo [0, 1] ou [-1, 1] é uma prática comum para manter a consistência metodológica e facilitar a convergência se outros modelos (como Redes Neurais) forem testados futuramente.

**Solução**:

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import numpy as np

def load_data(seed=42):
    X, y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False, parser='auto')
    y = y.astype(int)
    
    return train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)

X_train, X_test, y_train, y_test = load_data(42)


# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [2]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

def train_random_forest(X_train, y_train, seed=42):
    model = RandomForestClassifier(random_state=seed, n_jobs=-1)
    model.fit(X_train, y_train)
    return model

def train_adaboost(X_train, y_train, seed=42):
    model = AdaBoostClassifier(random_state=seed)
    model.fit(X_train, y_train)
    return model


# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [3]:
from sklearn.metrics import accuracy_score

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return acc


Para a avaliação do modelo, implementamos a função `evaluate` que utiliza a acurácia do `sklearn`. Esta métrica é adequada para este dataset equilibrado, fornecendo a porcentagem de classificações corretas no conjunto de teste.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [4]:
def run_pipeline(model_type="rf", seed=42):
    X_train, X_test, y_train, y_test = load_data(seed)
    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed)
    elif model_type == "ab":
        model = train_adaboost(X_train, y_train, seed)
    else:
        raise ValueError("model_type deve ser 'rf' ou 'ab'")
    
    acc = evaluate(model, X_test, y_test)
    return acc

acc_rf_demo = run_pipeline("rf", 42)
print(f"Acurácia RF Demo (Seed 42): {acc_rf_demo:.4f}")


Acurácia RF Demo (Seed 42): 0.9672


**Em qual profundidade começa o overfitting?**
O overfitting geralmente começa quando a árvore cresce o suficiente para capturar o ruído dos dados de treino. Em Random Forest, isso é mitigado pela amostragem aleatória, mas em profundidades muito altas (ou ilimitadas), cada árvore individual tende ao overfitting.

**Por que a árvore consegue 100% no treino quando max_depth=None?**
Porque ela pode continuar se expandindo até que cada folha contenha apenas instâncias de uma única classe, memorizando perfeitamente o conjunto de treinamento.

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Resposta**: O modelo que apresentou o melhor desempenho inicial foi o Random Forest. Isso ocorre por causa da sua capacidade de capturar relações complexas e não lineares nas imagens através de árvores profundas que atuam em conjunto. O AdaBoost, quando utiliza o classificador base padrão (árvores de profundidade 1), é incapaz de representar a complexidade das roupas/dígitos de forma tão eficiente quanto o RF no treinamento inicial.

**Solução**:

In [ ]:
from sklearn.metrics import classification_report

print("Executando pipeline para RF e AdaBoost...")
X_train, X_test, y_train, y_test = load_data(42)

rf_model = train_random_forest(X_train, y_train, 42)
rf_acc = evaluate(rf_model, X_test, y_test)

ab_model = train_adaboost(X_train, y_train, 42)
ab_acc = evaluate(ab_model, X_test, y_test)

print(f"Random Forest - Acurácia: {rf_acc:.4f}")
print(f"AdaBoost - Acurácia: {ab_acc:.4f}")

print("\nClassification Report RF:")
print(classification_report(y_test, rf_model.predict(X_test)))

print("\nClassification Report AB:")
print(classification_report(y_test, ab_model.predict(X_test)))

Executando pipeline para RF e AdaBoost...
Random Forest - Acurácia: 0.9672
AdaBoost - Acurácia: 0.6681

Classification Report RF:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98      1381
           1       0.98      0.98      0.98      1575
           2       0.96      0.97      0.97      1398
           3       0.96      0.96      0.96      1428
           4       0.97      0.96      0.96      1365
           5       0.97      0.96      0.96      1263
           6       0.97      0.98      0.98      1375
           7       0.97      0.97      0.97      1459
           8       0.96      0.96      0.96      1365
           9       0.94      0.94      0.94      1391

    accuracy                           0.97     14000
   macro avg       0.97      0.97      0.97     14000
weighted avg       0.97      0.97      0.97     14000


Classification Report AB:
              precision    recall  f1-score   support

           0       0.84      

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.


**Resposta:** Os resultados mudam ligeiramente devido à diferente amostragem dos dados, mas o experimento é perfeitamente reprodutível ao fixar a seed.

**Solução**:

In [6]:
acc_42 = run_pipeline("rf", 42)
acc_7 = run_pipeline("rf", 7)
print(f"Acurácia (Seed 42): {acc_42:.4f}")
print(f"Acurácia (Seed 7): {acc_7:.4f}")


Acurácia (Seed 42): 0.9672
Acurácia (Seed 7): 0.9681


# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting? 


**Resposta:** sim, pois o o resultado de treino foi maior que o de teste
- Qual modelo tende a sofrer mais com isso? 


**Resposta:** Árvores profundas individuais tendem ao overfitting, mas RF mitiga isso. AdaBoost pode sofrer overfitting com muitos estimadores fracos.

In [ ]:
rf_train_acc = rf_model.score(X_train, y_train)
rf_test_acc = rf_model.score(X_test, y_test)
print(f"RF Treino: {rf_train_acc:.4f}, Teste: {rf_test_acc:.4f}")

RF Treino: 1.0000, Teste: 0.9672


# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

**Resposta**: O AdaBoost é tecnicamente o modelo mais sensível. Isso acontece porque o Boosting é um processo sequencial onde cada nova árvore tenta corrigir os erros das anteriores. Se mudarmos o número de estimadores ou a qualidade do estimador base, o efeito cascata é muito maior do que no Random Forest, onde as árvores são independentes entre si e apenas "votam" no final.

In [9]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

for n in [50, 100]:
    rf = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    print(f"RF (n_estimators={n}): {rf.score(X_test, y_test):.4f}")

for n in [50, 100]:
    ab = AdaBoostClassifier(n_estimators=n, random_state=42)
    ab.fit(X_train, y_train)
    print(f"AB (n_estimators={n}): {ab.score(X_test, y_test):.4f}")


RF (n_estimators=50): 0.9639
RF (n_estimators=100): 0.9672
AB (n_estimators=50): 0.6681
AB (n_estimators=100): 0.7387


# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

### Respostas da Questão 9:

1. **A acurácia é suficiente?**
Não. A acurácia isolada pode ser enganosa, principalmente quando há desbalanceamento de classes, pois mostra apenas o desempenho global. Mesmo com datasets equilibrados como MNIST/Fashion, ela não revela quais classes estão sendo confundidas.

2. **Como garantir que o resultado não ocorreu por acaso?**
O controle de aleatoriedade com o parâmetro random_state garante reprodutibilidade. Ao fixar a seed, qualquer execução do código gera os mesmos resultados, permitindo verificação e comparação consistente.

3. **Problemas metodológicos:**
- **Falta de Tuning:** O uso de hiperparâmetros padrão (default) pode não extrair o potencial máximo dos modelos; uma busca em grade (Grid Search) seria ideal.
- **Avaliação Única:** O experimento baseia-se em uma única divisão treino/teste. Embora a semente seja fixa, o desempenho pode variar dependendo da "sorte" dessa divisão específica se o conjunto for pequeno.

4. **O pipeline é confiável?**
Sim. O pipeline mantém o conjunto de teste isolado até a avaliação final, evitando vazamento de dados e garantindo resultados mais confiáveis. Além disso, a organização modular e o uso de seeds tornam o processo reproduzível, claro e fácil de validar, aumentando a confiabilidade do experimento.